
# CNN Design Challenge

**Goal:** Design, train, and evaluate a custom **Sequential CNN** (Conv/Pool/BN/Dropout/Flatten/Linear only) on the provided Tiny‑ImageNet‑style dataset (64×64 RGB, 15 classes).  
This notebook is **self-contained** and aligned with DSII/AI techniques covered in class: reproducibility, normalization, augmentation, regularization, model selection, and transparent reporting.

- **Data:** use provided pickles `train-70_.pkl` and `validation-10_.pkl`.
- **Model:** fully custom **sequential** CNN (no pretrained nets, no residuals/attention).
- **Techniques:** reproducible seed, normalization, reasonable data augmentation, AdamW + LR scheduling, early stopping, dropout, BatchNorm, weight decay.
- **Outputs:** save best weights `checkpoints/model.pth` and `checkpoints/meta.json`.
- **Helpers:** `TinyCNN` class + `load_model(...)` + `predict(...)` in this notebook.
- **Reporting:** training/validation curves, final metrics, confusion matrix, sample predictions.


In [ ]:
# Powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
pip install -r requirements.txt



In [ ]:
# Linux/MacOS
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
pip install -r requirements.txt


In [ ]:
# Environment Setup and Imports
import os, sys, json, math, random, time, pickle
from pathlib import Path
from typing import Any, Dict, Tuple, List, Optional, Union

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

def set_global_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
set_global_seed(42)
print("Device:", DEVICE)
print("Python:", sys.version)
print("Torch:", torch.__version__)
try:
    import torchvision
    print("Torchvision:", torchvision.__version__)
except Exception as e:
    print("Torchvision not found:", e)


In [ ]:
# CONFIG 
# By default, I have it set to your current folder so everyone-else can clone-and-run.
# You can override via environment variables DATA_DIR, TRAIN_PKL, VAL_PKL.

DATA_DIR = os.environ.get("DATA_DIR", str(Path.cwd()))
TRAIN_PKL_NAME = os.environ.get("TRAIN_PKL", "train-70_.pkl")
VAL_PKL_NAME   = os.environ.get("VAL_PKL",   "validation-10_.pkl")

TRAIN_PKL = str(Path(DATA_DIR) / TRAIN_PKL_NAME)
VAL_PKL   = str(Path(DATA_DIR) / VAL_PKL_NAME)

# Fallback: if not found, try a ./data subfolder
if not (Path(TRAIN_PKL).exists() and Path(VAL_PKL).exists()):
    alt_dir = Path(DATA_DIR) / "data"
    if (alt_dir / TRAIN_PKL_NAME).exists() and (alt_dir / VAL_PKL_NAME).exists():
        TRAIN_PKL = str(alt_dir / TRAIN_PKL_NAME)
        VAL_PKL   = str(alt_dir / VAL_PKL_NAME)

# Friendly error if still missing
for p in [TRAIN_PKL, VAL_PKL]:
    if not Path(p).exists():
        raise FileNotFoundError(
            f"Missing data file: {p}\n"
            f"Place '{TRAIN_PKL_NAME}' and '{VAL_PKL_NAME}' next to the notebook "
            f"or set DATA_DIR/TRAIN_PKL/VAL_PKL env vars."
        )

BATCH_SIZE = 128
EPOCHS = 50
PATIENCE = 7
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
# Windows can be touchy with >0 workers in some school machines; adjust if needed.
NUM_WORKERS = 0 if os.name == "nt" else 2

IMG_SIZE = 64
NUM_CLASSES: Optional[int] = None  # will infer from labels

# Normalization (ImageNet-like) — you can recompute later if desired
NORM_MEAN = (0.485, 0.456, 0.406)
NORM_STD  = (0.229, 0.224, 0.225)

SAVE_DIR = Path.cwd() / "checkpoints"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
BEST_WEIGHTS_PATH = str(SAVE_DIR / "model.pth")
META_PATH = str(SAVE_DIR / "meta.json")
HIST_PATH = str(SAVE_DIR / "history.json")

print("DATA_DIR:", DATA_DIR)
print("TRAIN_PKL:", TRAIN_PKL)
print("VAL_PKL:", VAL_PKL)
print("SAVE_DIR:", SAVE_DIR)


In [ ]:

def _safe_pickle_load(path: str):
    # Load pickle robustly across Python/NumPy versions.
    last_err = None
    for enc in (None, 'latin1', 'bytes'):
        try:
            if enc is None:
                with open(path, 'rb') as f:
                    return pickle.load(f)
            else:
                with open(path, 'rb') as f:
                    return pickle.load(f, encoding=enc)
        except Exception as e:
            last_err = e
    raise last_err

def _to_chw(img: np.ndarray) -> np.ndarray:
    # Ensure array is CxHxW float32 in [0,1]. Accepts HxWxC or CxHxW, uint8/float.
    arr = np.asarray(img)
    if arr.ndim != 3:
        raise ValueError(f'Expected 3D array, got shape {arr.shape}')
    # If HWC, transpose to CHW
    if arr.shape[-1] == 3 and arr.shape[0] != 3:
        arr = np.transpose(arr, (2, 0, 1))
    # Convert dtype and scale if needed
    if arr.dtype != np.float32:
        arr = arr.astype(np.float32)
    if arr.max() > 1.5:
        arr = arr / 255.0
    return arr

class PickleDataset(Dataset):
    # Expects a .pkl containing either:
    #   - dict with keys "images" and "labels" (arrays/lists), optional "classes"/"class_to_idx"
    #   - list/tuple of (image, label) pairs
    # Images should be RGB 64x64; function tries to coerce CHW and float32.
    def __init__(self, pkl_path: str, transform=None):
        self.pkl_path = pkl_path
        self.data_raw = _safe_pickle_load(pkl_path)
        self.transform = transform

        if isinstance(self.data_raw, dict):
            if 'images' in self.data_raw and 'labels' in self.data_raw:
                self.images = self.data_raw['images']
                self.labels = self.data_raw['labels']
                self.class_to_idx = self.data_raw.get('class_to_idx', None)
                self.classes = self.data_raw.get('classes', None)
            else:
                raise ValueError(f'Unsupported dict keys: {list(self.data_raw.keys())}')
        elif isinstance(self.data_raw, (list, tuple)):
            self.images, self.labels = zip(*self.data_raw)
            self.class_to_idx, self.classes = None, None
        else:
            raise ValueError(f'Unsupported pickle format: {type(self.data_raw)}')

        self.num_samples = len(self.labels)
        self.unique_labels = sorted(set(int(x) for x in self.labels))
        self.num_classes = len(self.unique_labels)

    def __len__(self): return self.num_samples

    def __getitem__(self, idx: int):
        img = self.images[idx]
        lab = self.label_to_idx[int(self.labels[idx])]  # remapped 
        img = _to_chw(img)
        img = torch.from_numpy(img)  # CxHxW in [0,1]
        if self.transform is not None:
            img = self.transform(img)
        return img, lab


In [ ]:

# EDA transforms
from torchvision import transforms
eda_transform = transforms.Compose([transforms.ConvertImageDtype(torch.float)])

train_ds_raw = PickleDataset(TRAIN_PKL, transform=eda_transform)
val_ds_raw   = PickleDataset(VAL_PKL,   transform=eda_transform)

if NUM_CLASSES is None:
    NUM_CLASSES = max(train_ds_raw.num_classes, val_ds_raw.num_classes)

print('Train:', len(train_ds_raw), 'Val:', len(val_ds_raw), '| Classes:', NUM_CLASSES)

# Class distribution
def label_counts(ds: PickleDataset):
    import collections
    c = collections.Counter([int(x) for x in ds.labels])
    return c

train_counts = label_counts(train_ds_raw)
val_counts   = label_counts(val_ds_raw)
print('Train label counts (first 10):', list(train_counts.items())[:10])
print('Val label counts (first 10):', list(val_counts.items())[:10])

# Sample grid
def show_grid(ds: PickleDataset, n: int = 16):
    import numpy as np
    n = min(n, len(ds))
    idxs = np.linspace(0, len(ds)-1, n, dtype=int)
    cols = int(np.sqrt(n))
    rows = int(np.ceil(n/cols))
    fig = plt.figure(figsize=(cols*2, rows*2))
    for i, idx in enumerate(idxs):
        x, y = ds[idx]
        x_np = x.numpy().transpose(1,2,0)  # HWC
        ax = fig.add_subplot(rows, cols, i+1)
        ax.imshow(x_np)
        ax.set_title(f'y={y}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# show_grid(train_ds_raw, n=16)  # uncomment to visualize


In [ ]:

# testing / optional: compute dataset mean/std (on subset for speed)
def compute_mean_std(dataset: Dataset, samples: int = 2000, seed: int = 42):
    set_global_seed(seed)
    import numpy as np, torch
    idxs = np.random.choice(len(dataset), size=min(samples, len(dataset)), replace=False)
    s = []
    for i in idxs:
        x, _ = dataset[i]
        s.append(x.unsqueeze(0))
    X = torch.cat(s, dim=0)  # N,C,H,W
    mean = X.mean(dim=(0,2,3)).tolist()
    std  = X.std(dim=(0,2,3)).tolist()
    return tuple(mean), tuple(std)

# NORM_MEAN, NORM_STD = compute_mean_std(train_ds_raw, samples=4000)
# print('Dataset mean/std:', NORM_MEAN, NORM_STD)


In [ ]:

# Final transforms + loaders
train_transform = transforms.Compose([
    transforms.ConvertImageDtype(torch.float),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(IMG_SIZE, padding=4),
    transforms.RandomRotation(10),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

val_transform = transforms.Compose([
    transforms.ConvertImageDtype(torch.float),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

train_ds = PickleDataset(TRAIN_PKL, transform=train_transform)
val_ds   = PickleDataset(VAL_PKL,   transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print('Dataloaders ready.')



## Model Design Rationale (DSII-aligned)
- 4 Conv blocks (32→64→128→256), each with BN and ReLU, then MaxPool + Dropout.
- Classifier: Flatten → Linear(256×4×4→512) → ReLU → Dropout → Linear(→NUM_CLASSES).
- Optimizer: AdamW with weight decay; Scheduler: ReduceLROnPlateau.
- Early stopping based on validation loss; AMP mixed precision; gradient clipping.


In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, num_classes: int = 15):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),   # 64 -> 32
            nn.Dropout(0.10),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),   # 32 -> 16
            nn.Dropout(0.15),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),   # 16 -> 8
            nn.Dropout(0.20),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),   # 8 -> 4
            nn.Dropout(0.25),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*4*4, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes), 
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, a=math.sqrt(5))
                if m.bias is not None:
                    fan_in = m.weight.size(1)
                    bound = 1 / math.sqrt(fan_in)
                    nn.init.uniform_(m.bias, -bound, bound)

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = TinyCNN(num_classes=NUM_CLASSES).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'Total params: {total_params:,}')


In [ ]:

from dataclasses import dataclass

@dataclass
class History:
    train_loss: List[float]
    val_loss: List[float]
    train_acc: List[float]
    val_acc: List[float]

def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()

@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        acc = accuracy_from_logits(logits, yb)
        total_loss += loss.item()
        total_acc += acc
        n_batches += 1
    return total_loss / n_batches, total_acc / n_batches

def train(model, train_loader, val_loader, epochs=EPOCHS, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, patience=PATIENCE):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)
    loss_fn = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=='cuda'))

    best_val = float('inf')
    best_epoch = -1
    hist = History([], [], [], [])

    for epoch in range(1, epochs+1):
        model.train()
        epoch_loss = 0.0
        epoch_acc = 0.0
        n_batches = 0
        start = time.time()

        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE=='cuda')):
                logits = model(xb)
                loss = loss_fn(logits, yb)
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            scaler.step(optimizer)
            scaler.update()

            acc = accuracy_from_logits(logits, yb)
            epoch_loss += loss.item()
            epoch_acc += acc
            n_batches += 1

        train_loss = epoch_loss / n_batches
        train_acc = epoch_acc / n_batches
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)
        scheduler.step(val_loss)

        hist.train_loss.append(train_loss)
        hist.val_loss.append(val_loss)
        hist.train_acc.append(train_acc)
        hist.val_acc.append(val_acc)

        dur = time.time() - start
        print(f"Epoch {epoch:02d}/{epochs} | {dur:.1f}s  Train: loss {train_loss:.4f} acc {train_acc:.4f}  |  Val: loss {val_loss:.4f} acc {val_acc:.4f}")

        if val_loss < best_val:
            best_val = val_loss
            best_epoch = epoch
            torch.save(model.state_dict(), BEST_WEIGHTS_PATH)
            meta = {
                'num_classes': NUM_CLASSES,
                'norm_mean': NORM_MEAN,
                'norm_std': NORM_STD,
                'img_size': IMG_SIZE,
                'saved_at_epoch': epoch,
                'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
            }
            with open(META_PATH, 'w') as f:
                json.dump(meta, f, indent=2)
            print(f"  ↳ Saved new best weights to {BEST_WEIGHTS_PATH}")

        if epoch - best_epoch >= patience:
            print(f"Early stopping at epoch {epoch} (no improvement for {patience} epochs). Best epoch was {best_epoch}.")
            break

    with open(HIST_PATH, 'w') as f:
        json.dump({
            'train_loss': hist.train_loss,
            'val_loss': hist.val_loss,
            'train_acc': hist.train_acc,
            'val_acc': hist.val_acc
        }, f, indent=2)

    return hist


In [ ]:

# === TRAIN ===
history = train(model, train_loader, val_loader)
print('Training complete. Best weights saved to:', BEST_WEIGHTS_PATH)


In [ ]:

# === LEARNING CURVES ===
with open(HIST_PATH, 'r') as f:
    h = json.load(f)

fig = plt.figure(figsize=(6,4))
plt.plot(h['train_loss'], label='train_loss')
plt.plot(h['val_loss'], label='val_loss')
plt.title('Loss vs Epoch')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

fig = plt.figure(figsize=(6,4))
plt.plot(h['train_acc'], label='train_acc')
plt.plot(h['val_acc'], label='val_acc')
plt.title('Accuracy vs Epoch')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# === EVALUATION ===
best = TinyCNN(num_classes=NUM_CLASSES).to(DEVICE)
best.load_state_dict(torch.load(BEST_WEIGHTS_PATH, map_location=DEVICE))

val_loss, val_acc = evaluate(best, val_loader, nn.CrossEntropyLoss())
print(f'Best checkpoint -> val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f}')

# Collect predictions
all_preds, all_labels = [], []
best.eval()
with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(DEVICE, non_blocking=True)
        logits = best(xb)
        preds = logits.argmax(dim=1).cpu().numpy().tolist()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy().tolist())

print('\nClassification report:')
print(classification_report(all_labels, all_preds, digits=4))

cm = confusion_matrix(all_labels, all_preds, labels=sorted(set(all_labels)))
fig = plt.figure(figsize=(6,6))
plt.imshow(cm, interpolation='nearest')
plt.title('Validation Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.colorbar()
plt.tight_layout()
plt.show()

# Show a few validation images with predictions vs truth
def show_val_samples(ds: Dataset, preds: List[int], labels: List[int], n: int = 12):
    cols = 4
    rows = int(np.ceil(n/cols))
    fig = plt.figure(figsize=(cols*2.2, rows*2.2))
    step = max(1, len(ds)//n)
    idxs = list(range(0, len(ds), step))[:n]
    for i, idx in enumerate(idxs):
        x, y = ds[idx]
        mean = torch.tensor(NORM_MEAN).view(3,1,1)
        std  = torch.tensor(NORM_STD).view(3,1,1)
        x_disp = x*std + mean
        x_np = x_disp.numpy().transpose(1,2,0)
        ax = fig.add_subplot(rows, cols, i+1)
        ax.imshow(np.clip(x_np, 0, 1))
        ax.set_title(f'pred={preds[idx]} / true={labels[idx]}', fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

preds_in_order = []
labels_in_order = []
best.eval()
with torch.no_grad():
    for i in range(len(val_ds)):
        x, y = val_ds[i]
        logits = best(x.unsqueeze(0).to(DEVICE))
        pred = int(logits.argmax(dim=1).item())
        preds_in_order.append(pred)
        labels_in_order.append(y)

show_val_samples(val_ds, preds_in_order, labels_in_order, n=12)


In [ ]:

# === SUBMISSION HELPERS ===
def load_model(weights_path: str, num_classes: Optional[int] = None) -> TinyCNN:
    if num_classes is None and os.path.exists(META_PATH):
        with open(META_PATH, 'r') as f:
            m = json.load(f)
        num_classes = int(m.get('num_classes', NUM_CLASSES if NUM_CLASSES else 15))
    elif num_classes is None:
        num_classes = NUM_CLASSES if NUM_CLASSES else 15

    model = TinyCNN(num_classes=num_classes)
    state = torch.load(weights_path, map_location='cpu')
    model.load_state_dict(state, strict=True)
    model.eval()
    return model

@torch.no_grad()
def predict(model_or_weights, data):
    # Returns integer class predictions for numpy arrays or torch loaders/datasets
    if isinstance(model_or_weights, str):
        model = load_model(model_or_weights)
    else:
        model = model_or_weights
    model.eval()

    def _predict_from_array(arr: np.ndarray):
        if arr.ndim != 4:
            raise ValueError(f'Expected 4D array, got {arr.shape}')
        if arr.shape[-1] == 3:  # NHWC -> NCHW
            arr = np.transpose(arr, (0,3,1,2))
        x = arr.astype('float32')
        if x.max() > 1.5:
            x = x / 255.0
        x = torch.from_numpy(x)
        mean = torch.tensor(NORM_MEAN).view(1,3,1,1)
        std  = torch.tensor(NORM_STD).view(1,3,1,1)
        x = (x - mean) / std
        logits = model(x)
        return logits.argmax(dim=1).cpu().numpy().tolist()

    if isinstance(data, np.ndarray):
        return _predict_from_array(data)
    if isinstance(data, list):
        arr = np.stack([np.asarray(x) for x in data], axis=0)
        return _predict_from_array(arr)
    if isinstance(data, DataLoader):
        loader = data
    elif isinstance(data, Dataset):
        loader = DataLoader(data, batch_size=128, shuffle=False, num_workers=NUM_WORKERS)
    else:
        raise ValueError('Unsupported data type for predict(...)')

    preds_all = []
    for xb, _ in loader:
        logits = model(xb)
        preds = logits.argmax(dim=1).cpu().numpy().tolist()
        preds_all.extend(preds)
    return preds_all
